## Feature Extraction
Building on the insights from our Exploratory Data Analysis (EDA), this notebook shifts focus to preprocessing and extracting raw features from our ingested data. Our primary objective is to build the baseline features for the downstream experiments.

#### Imports

In [ ]:
import pandas as pd
import numpy as np
from pandas.tseries.holiday import USFederalHolidayCalendar
from pandas.io import parsers


In [9]:
## load the data
df= pd.read_csv(
    "../data/raw/ercot_south_central_raw.csv",
    parse_dates=["timestamp", "timestamp_central"]) 
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df["timestamp_central"] = pd.to_datetime(df["timestamp_central"], utc=True).dt.tz_convert("America/Chicago")
df = df.sort_values("timestamp").reset_index(drop=True)

print(df.shape)
df.head()

(92879, 9)


,timestamp,timestamp_central,actual_load_mw,temp_c,humidity_pct,precip_mm,tmax,tmin,tavg
0,2016-01-01 00:00:00+00:00,2015-12-31 18:00:00-06:00,6848.58,10.902697,60.428911,0.0,10.902697,9.96666,10.465636
1,2016-01-01 01:00:00+00:00,2015-12-31 19:00:00-06:00,6599.73,10.714651,60.498833,0.0,10.902697,9.96666,10.465636
2,2016-01-01 02:00:00+00:00,2015-12-31 20:00:00-06:00,6357.69,10.595334,61.405271,0.0,10.902697,9.96666,10.465636
3,2016-01-01 03:00:00+00:00,2015-12-31 21:00:00-06:00,6120.83,10.409551,61.834183,0.0,10.902697,9.96666,10.465636
4,2016-01-01 04:00:00+00:00,2015-12-31 22:00:00-06:00,5882.63,10.204923,62.263094,0.0,10.902697,9.96666,10.465636


In [10]:
# Reindex to complete hourly UTC range
# Theres one missing hour from our gridstatus API (2016-03-16 3:00 -> 05:00 UTC)
df = df.set_index("timestamp")
full_range = pd.date_range(df.index.min(), df.index.max(), freq="h", tz="UTC")

n_before = len(df)
df = df.reindex(full_range)
df.index.name = "timestamp"
n_after = len(df)

print(f"Rows before reindex: {n_before}")
print(f"Rows after reindex : {n_after}  (inserted {n_after - n_before} gap row(s))")

# recompute timestamp_central from the now-complete index, rather than trusting the old column
df["timestamp_central"] = df.index.tz_convert("America/Chicago")
df = df.reset_index()

df[df["actual_load_mw"].isna()]

Rows before reindex: 92879
Rows after reindex : 92880  (inserted 1 gap row(s))


,timestamp,timestamp_central,actual_load_mw,temp_c,humidity_pct,precip_mm,tmax,tmin,tavg
1804,2016-03-16 04:00:00+00:00,2016-03-15 23:00:00-05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##### Add calender features from `timestamp_central`

In [11]:
df["hour"] = df["timestamp_central"].dt.hour
df["dayofweek"] = df["timestamp_central"].dt.dayofweek   # 0=Mon ... 6=Sun
df["month"] = df["timestamp_central"].dt.month
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

df[["timestamp", "timestamp_central", "hour", "dayofweek", "month", "is_weekend"]].head()

,timestamp,timestamp_central,hour,dayofweek,month,is_weekend
0,2016-01-01 00:00:00+00:00,2015-12-31 18:00:00-06:00,18,3,12,0
1,2016-01-01 01:00:00+00:00,2015-12-31 19:00:00-06:00,19,3,12,0
2,2016-01-01 02:00:00+00:00,2015-12-31 20:00:00-06:00,20,3,12,0
3,2016-01-01 03:00:00+00:00,2015-12-31 21:00:00-06:00,21,3,12,0
4,2016-01-01 04:00:00+00:00,2015-12-31 22:00:00-06:00,22,3,12,0


##### Holiday Flag

In [13]:
cal = USFederalHolidayCalendar()

start = df["timestamp_central"].min().tz_localize(None)
end   = df["timestamp_central"].max().tz_localize(None)
holidays = cal.holidays(start=start, end=end)

central_date = df["timestamp_central"].dt.normalize().dt.tz_localize(None)
df["is_holiday"] = central_date.isin(holidays).astype(int)

df["is_holiday"].value_counts()

is_holiday
0    90216
1     2664
Name: count, dtype: int64

##### Load Lag features

In [14]:
df["load_lag_24"]  = df["actual_load_mw"].shift(24)
df["load_lag_168"] = df["actual_load_mw"].shift(168)

# manual spot-check: row i's load_lag_24 should equal row i-24's actual_load_mw
i = 500
print(df.loc[i, "load_lag_24"] == df.loc[i - 24, "actual_load_mw"])
print(df.loc[i, "load_lag_168"] == df.loc[i - 168, "actual_load_mw"])

True
True


##### Drop warm-up / gap rows

In [15]:
before = len(df)
df_clean = df.dropna(subset=["actual_load_mw", "load_lag_24", "load_lag_168"]).reset_index(drop=True)
after = len(df_clean)

print(f"Dropped {before - after} rows (lag warm-up + the one gap hour's downstream effects)")
print(df_clean.isna().sum())

Dropped 171 rows (lag warm-up + the one gap hour's downstream effects)
timestamp            0
timestamp_central    0
actual_load_mw       0
temp_c               0
humidity_pct         0
precip_mm            0
tmax                 0
tmin                 0
tavg                 0
hour                 0
dayofweek            0
month                0
is_weekend           0
is_holiday           0
load_lag_24          0
load_lag_168         0
dtype: int64


In [16]:
# sanity check the final baseline frame for statistical anomalies
baseline_cols = [
    "timestamp", "timestamp_central", "actual_load_mw",
    "temp_c", "humidity_pct", "precip_mm", "tmax", "tmin", "tavg",
    "hour", "dayofweek", "month", "is_weekend", "is_holiday",
    "load_lag_24", "load_lag_168",
]
df_clean = df_clean[baseline_cols]

print(df_clean.shape)
df_clean.describe()

(92709, 16)


,actual_load_mw,temp_c,humidity_pct,precip_mm,tmax,tmin,tavg,hour,dayofweek,month,is_weekend,is_holiday,load_lag_24,load_lag_168
count,92709.000000,92709.000000,92709.000000,92709.000000,92709.000000,92709.000000,92709.000000,92709.000000,92709.000000,92709.000000,92709.000000,92709.000000,92709.000000,92709.000000
mean,7636.678490,21.438560,66.731596,0.126113,26.911017,16.631112,21.438640,11.499693,3.000076,6.397416,0.285787,0.028476,7634.969520,7626.204331
std,2155.090725,8.282736,20.986344,0.600886,7.703418,7.652955,7.444034,6.922145,2.000240,3.427419,0.451791,0.166330,2153.869156,2146.297186
min,3772.190000,-13.661563,5.958575,0.000000,-3.314912,-13.661563,-8.743937,0.000000,0.000000,1.000000,0.000000,0.000000,3772.190000,3772.190000
25%,6033.820000,15.966442,50.089390,0.000000,21.817581,10.806338,16.245108,6.000000,1.000000,3.000000,0.000000,0.000000,6033.160000,6030.720000
50%,7167.600000,22.586891,68.480811,0.000000,28.067131,18.266989,22.622363,11.000000,3.000000,6.000000,0.000000,0.000000,7166.050000,7158.860000
75%,8864.680000,27.221040,85.271809,0.002651,33.000249,23.379833,27.603846,17.000000,5.000000,9.000000,1.000000,0.000000,8862.540000,8848.520000
max,15564.030000,41.527919,100.000000,18.712918,41.527919,28.569676,34.423739,23.000000,6.000000,12.000000,1.000000,1.000000,15564.030000,15564.030000


In [19]:
df_clean.to_parquet('../data/interim/df_core_features.parquet', index=False)
